# AE2: Obtención y reconocimiento del dataset Goodreads Books

Primera etapa del proyecto integrador. Este notebook obtiene el dataset desde Hugging Face, lo carga en un DataFrame de Pandas, lo describe brevemente y genera automáticamente su resumen en `AE2/resultados/resumen_dataframe.txt`.

**Escenario hipotético.** Una librería planea rediseñar su exhibición en dos áreas estratégicas: *Lecturas Ágiles* (libros de volumen delgado o mediano) y *Grandes Clásicos y Sagas* (obras de gran extensión). Para decidir qué porcentaje de espacio asignar a cada área y qué editoriales e idiomas priorizar en las compras importadas, requiere un diagnóstico descriptivo de los libros destacados (calificación promedio ≥ 3.5) en Goodreads.

**Preguntas de negocio, variables y resultados esperados:**

| Pregunta | Variables | Resultado esperado |
|---|---|---|
| ¿Cómo se distribuyen los libros destacados según su extensión? | `average_rating`, `pages`, `book_id`, `work_id` | Conteos y porcentajes por categoría de páginas, separando ediciones y obras. |
| ¿Qué editoriales e idiomas tienen más títulos destacados en cada grupo? | Categoría de páginas, `publisher`, `language`, `book_id`, `work_id` | Rankings con conteos de ediciones y obras únicas; no representan ventas. |
| ¿Cómo varían la calificación y la participación de lectores entre grupos? | `average_rating`, `ratings_count`, `reviews_count`, `work_id` | Distribuciones y percentiles; las métricas de lectores no se suman por edición. |
| ¿Qué proporción de registros pertenece a una serie según su extensión? | `is_part_of_series`, `series`, categoría de páginas | Conteos y porcentajes de registros en series por grupo. |

Goodreads no contiene inventario, ventas ni espacio físico de una librería; los resultados son descriptivos y no prueban demanda comercial.


Flujo:
1. Configuración de rutas y parámetros del dataset.
2. Obtención del dataset con `HFDatasetLoader` (`src/hf_loader.py`).
3. Descripción básica: registros, dimensiones, columnas, tipos y valores nulos.
4. Qué representa cada fila y qué variables respaldan la necesidad.
5. Generación del resumen `.txt` con `Reportes` (`src/reportes.py`).
6. Exploración de la calidad de páginas, umbrales candidatos y diferencias entre ediciones y obras.

## 1. Configuración

In [19]:
import os
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Ruta dinámica hacia la raíz del repositorio (la carpeta que contiene 'src')
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from datasets import disable_progress_bars

from hf_loader import HFDatasetLoader
from reportes import Reportes

# Sin barras de progreso para que la salida guardada del notebook quede limpia
disable_progress_bars()

# Parámetros del dataset utilizado
DATASET_ID = "pfaha/goodreads-books"
DATASET_URL = f"https://huggingface.co/datasets/{DATASET_ID}"
CONFIG = "default"
SPLIT = "train"
# El dataset crece con nuevos checkpoints; se fija la revisión consultada para reproducir los resultados
REVISION = "43b3198cd7ec9c142d56181f1a09ed9e8c112610"
FECHA_CONSULTA = "2026-10-24"

RUTA_RESUMEN = ROOT / "AE2" / "resultados" / "resumen_dataframe.txt"

print(f"Raíz del proyecto: {ROOT.name}")
print(f"Dataset: {DATASET_ID} | config: {CONFIG} | split: {SPLIT} | revisión: {REVISION[:7]}")

Raíz del proyecto: RepoMachine_ProInteg3
Dataset: pfaha/goodreads-books | config: default | split: train | revisión: 43b3198


## 2. Obtención del dataset

Se descarga el split `train` completo (no se usa una muestra). La descarga es pública y no requiere token; los archivos quedan en la caché local de Hugging Face y no se guardan en el repositorio.

In [20]:
loader = HFDatasetLoader(
    dataset_name=DATASET_ID,
    config=CONFIG,
    split=SPLIT,
    revision=REVISION,
)
df = loader.load()

print("Dataset cargado en un DataFrame de Pandas.")
loader.info()

Dataset cargado en un DataFrame de Pandas.


{'dataset': 'pfaha/goodreads-books',
 'config': 'default',
 'split': 'train',
 'revision': '43b3198cd7ec9c142d56181f1a09ed9e8c112610',
 'filas': 500000,
 'columnas': 24}

## 3. Descripción básica

### 3.1 Algunos registros

In [21]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)
df.head(5)

,title,contributors,first_author,description,genres,series,format,pages,language,asin,isbn10,isbn13,publisher,publication_date,average_rating,ratings_count,reviews_count,work_id,catalog_guideline_violation,book_id,url,num_editions,has_more_editions,is_part_of_series
0,La doctrina secreta. Vol I. Cosmogen...,[{'name': 'Helena Petrovna Blavatsky...,Helena Petrovna Blavatsky,"Esta es una obra inteligente, origin...","[Occult, Spirituality, Nonfiction, P...","[{'name': 'The Secret Doctrine', 'po...",Paperback,319.0,Spanish; Castilian,950171103X,950171103X,9789501711035,Editorial Kier,2005-12-15,4.16,255.0,22.0,44066165.0,False,7282453,https://www.goodreads.com/book/show/...,124.0,True,True
1,Clement Marot and the Inflections of...,"[{'name': 'Robert Griffin', 'url': '...",Robert Griffin,This title is part of UC Press's Voi...,[],[],Hardcover,NaN,English,0520025865,0520025865,9780520025868,University of California Press,1974-01-01,0.00,0.0,0.0,1320719.0,False,1331224,https://www.goodreads.com/book/show/...,2.0,True,False
2,How to Market Legal Services,"[{'name': 'Robert W Denney', 'url': ...",Robert W Denney,NaN,[],[],Hardcover,278.0,English,0442219806,0442219806,9780442219802,Van Nostrand Reinhold Company,1984-09-01,0.00,0.0,0.0,3956063.0,False,3910678,https://www.goodreads.com/book/show/...,2.0,True,False
3,Personality and Intellectual Competence,"[{'name': 'Tomas Chamorro-Premuzic',...",Tomas Chamorro-Premuzic,This book provides a comprehensive s...,"[Psychology, Reference]",[],Hardcover,214.0,English,0805851364,0805851364,9780805851366,Psychology Press,2014-06-19,3.88,8.0,0.0,580945.0,False,2027361,https://www.goodreads.com/book/show/...,8.0,True,False
4,Studies in the Text History Of the L...,"[{'name': 'Ben Edwin Perry', 'url': ...",Ben Edwin Perry,"English, Greek",[],[],Paperback,272.0,English,0891305343,0891305343,9780891305347,Oxford University Press,1981-05-01,0.00,0.0,0.0,3997874.0,False,3952223,https://www.goodreads.com/book/show/...,4.0,True,False


### 3.2 Dimensiones

In [22]:
filas, columnas = df.shape
print(f"Filas: {filas:,} | Columnas: {columnas}")

Filas: 500,000 | Columnas: 24


### 3.3 Columnas y tipos de datos

In [23]:
tipos = pd.DataFrame({
    "columna": df.columns,
    "tipo_pandas": df.dtypes.astype(str).values,
    "ejemplo": [repr(df[c].dropna().iloc[0])[:50] if df[c].notna().any() else None for c in df.columns],
})
tipos

,columna,tipo_pandas,ejemplo
0,title,str,'La doctrina secreta. Vol I. Cosmoge...
1,contributors,object,array([{'name': 'Helena Petrovna Bla...
2,first_author,str,'Helena Petrovna Blavatsky'
3,description,str,"'Esta es una obra inteligente, origi..."
4,genres,object,"array(['Occult', 'Spirituality', 'No..."
5,series,object,array([{'name': 'The Secret Doctrine...
6,format,str,'Paperback'
7,pages,float64,np.float64(319.0)
8,language,str,'Spanish; Castilian'
9,asin,str,'950171103X'


### 3.4 Valores nulos por columna

Además de los nulos, se cuentan los valores vacíos (texto `''` o lista vacía), porque el scraping suele dejar campos vacíos en lugar de nulos.

In [24]:
reportes = Reportes()
calidad = pd.DataFrame({
    "nulos": df.isna().sum(),
    "vacios": [reportes._contar_vacios(df[c]) for c in df.columns],
})
calidad["porcentaje_nulos_o_vacios"] = ((calidad["nulos"] + calidad["vacios"]) / len(df) * 100).round(2)
calidad

,nulos,vacios,porcentaje_nulos_o_vacios
title,0,0,0.00
contributors,0,160,0.03
first_author,160,0,0.03
description,65575,14,13.12
genres,0,307620,61.52
series,0,437717,87.54
format,5107,2258,1.47
pages,38992,0,7.80
language,19787,0,3.96
asin,2909,0,0.58


## 4. Qué representa cada fila y qué variables respaldan la necesidad

**Unidad de análisis.** Cada fila es un libro (edición) de Goodreads identificado por `book_id`, obtenido por muestreo aleatorio de identificadores. Los campos bibliográficos (`pages`, `publisher`, `language`, `format`) describen esa edición, mientras que `average_rating`, `ratings_count` y `reviews_count` se agregan a nivel de la obra (todas sus ediciones), según la ficha del dataset.

Observaciones de la revisión:
- `pages`, `ratings_count`, `reviews_count`, `work_id` y `num_editions` figuran como `int64` en la ficha, pero Pandas las carga como `float64` porque contienen nulos.
- En los registros revisados, una obra sin calificaciones (`ratings_count = 0`) aparece con `average_rating = 0.00`. Ese 0 indica ausencia de calificaciones, no una calificación baja; no afecta al filtro ≥ 3.5.
- `genres` y `series` usan listas vacías en lugar de nulos.

**Variables que respaldan la necesidad:**

| Variable | Uso en la necesidad |
|---|---|
| `pages` | Clasificar los libros en *Lecturas Ágiles* o *Grandes Clásicos y Sagas* según su extensión. |
| `average_rating` | Delimitar los libros destacados (≥ 3.5) y comparar la valoración entre grupos. |
| `ratings_count`, `reviews_count` | Medir la participación de los lectores en cada grupo. |
| `publisher` | Identificar las editoriales que concentran más títulos destacados. |
| `language` | Identificar los idiomas que conviene priorizar en las compras importadas. |
| `is_part_of_series`, `series` | Caracterizar las sagas dentro del área de grandes obras. |
| `format`, `genres` | Variables de apoyo para describir cada área. |

### Dimensionamiento preliminar

El conteo siguiente solo dimensiona la población de interés; no es el análisis de la etapa 2. Los cortes son **provisionales**: un número de páginas válido entre 1 y 5,000 (para excluir valores anómalos como 0 o cifras de millones) y un umbral de 350 páginas entre las dos áreas, cercano al tercer cuartil observado. Ambos se validarán en la siguiente etapa.

In [25]:
PAGINAS_MIN, PAGINAS_MAX = 1, 5000   # rango provisional de páginas válidas
UMBRAL_AGILES = 350                  # umbral provisional entre las dos áreas

destacados = df[df["average_rating"] >= 3.5]
validos = destacados[destacados["pages"].between(PAGINAS_MIN, PAGINAS_MAX)]
area = pd.cut(
    validos["pages"],
    bins=[PAGINAS_MIN - 1, UMBRAL_AGILES, PAGINAS_MAX],
    labels=["Lecturas Ágiles", "Grandes Clásicos y Sagas"],
)

print(f"Libros con calificación >= 3.5: {len(destacados):,} de {len(df):,}")
print(f"De ellos, con páginas entre {PAGINAS_MIN} y {PAGINAS_MAX}: {len(validos):,}")
informado = lambda serie: serie.fillna("").str.strip().ne("").sum()
print(f"Con editorial informada: {informado(validos['publisher']):,}")
print(f"Con idioma informado: {informado(validos['language']):,}\n")
area.value_counts().rename("libros").to_frame().assign(
    porcentaje=lambda t: (t["libros"] / t["libros"].sum() * 100).round(1)
)

Libros con calificación >= 3.5: 276,152 de 500,000
De ellos, con páginas entre 1 y 5000: 235,104
Con editorial informada: 228,189
Con idioma informado: 229,528



,libros,porcentaje
pages,,
Lecturas Ágiles,175788,74.8
Grandes Clásicos y Sagas,59316,25.2


## 5. Generación automática del resumen `.txt`

El resumen se genera a partir del mismo DataFrame `df` revisado arriba. Se guarda en UTF-8 en una ruta relativa del proyecto; la carpeta se crea si no existe y el archivo se sobrescribe en cada ejecución.

In [26]:
metadatos = {
    "Dataset": DATASET_ID,
    "Enlace": DATASET_URL,
    "Configuración": CONFIG,
    "Split": SPLIT,
    "Revisión": REVISION,
    "Fecha de consulta": FECHA_CONSULTA,
    "Muestra": "No; se resume el split train completo de la revisión indicada",
}

ruta_resumen = reportes.guardar_resumen_dataframe(df, RUTA_RESUMEN, metadatos)
print(f"Resumen generado en: {ruta_resumen.relative_to(ROOT).as_posix()}\n")

# Verificación: se vuelve a leer el archivo generado
print(ruta_resumen.read_text(encoding="utf-8"))

Resumen generado en: AE2/resultados/resumen_dataframe.txt

RESUMEN DEL DATAFRAME
Dataset: pfaha/goodreads-books
Enlace: https://huggingface.co/datasets/pfaha/goodreads-books
Configuración: default
Split: train
Revisión: 43b3198cd7ec9c142d56181f1a09ed9e8c112610
Fecha de consulta: 2026-10-24
Muestra: No; se resume el split train completo de la revisión indicada

Filas: 500000
Columnas: 24

Columnas y tipos:
- title: str
- contributors: object
- first_author: str
- description: str
- genres: object
- series: object
- format: str
- pages: float64
- language: str
- asin: str
- isbn10: str
- isbn13: str
- publisher: str
- publication_date: str
- average_rating: float64
- ratings_count: float64
- reviews_count: float64
- work_id: float64
- catalog_guideline_violation: bool
- book_id: str
- url: str
- num_editions: float64
- has_more_editions: bool
- is_part_of_series: bool

Valores nulos por columna:
- title: 0
- contributors: 0
- first_author: 160
- description: 65575
- genres: 0
- series: 0

## 6. Exploración para definir la preparación del análisis

Esta sección **no modifica `df`**: conserva la carga original como referencia y construye subconjuntos de trabajo para cuantificar criterios antes de transformar. El objetivo es decidir con evidencia cómo clasificar la extensión, qué registros quedan fuera por calidad de páginas y cómo distinguir ediciones (`book_id`) de obras (`work_id`). Los cortes comparados son escenarios exploratorios, no decisiones definitivas.

La ficha del dataset indica que páginas, editorial e idioma describen una edición, mientras que las métricas de valoración y reseñas se agregan a nivel de obra. Por eso se muestran conteos de registros y obras por separado y no se suman `ratings_count` ni `reviews_count` por edición.

In [27]:
import numpy as np
# `df` permanece como la carga original; todos los filtros siguientes crean copias.
df_original = df

campos_necesidad = [
    "book_id", "work_id", "pages", "average_rating", "ratings_count",
    "reviews_count", "publisher", "language", "is_part_of_series",
]
calidad_campos = pd.DataFrame({
    "nulos": df_original[campos_necesidad].isna().sum(),
    "vacios": [reportes._contar_vacios(df_original[c]) for c in campos_necesidad],
    "valores_distintos_no_nulos": [df_original[c].nunique(dropna=True) for c in campos_necesidad],
})
calidad_campos["porcentaje_nulos_o_vacios"] = (
    (calidad_campos["nulos"] + calidad_campos["vacios"]) / len(df_original) * 100
).round(2)
calidad_campos

,nulos,vacios,valores_distintos_no_nulos,porcentaje_nulos_o_vacios
book_id,0,0,500000,0.00
work_id,253,0,461678,0.05
pages,38992,0,2306,7.80
average_rating,258,0,318,0.05
ratings_count,258,0,28289,0.05
reviews_count,258,0,10214,0.05
publisher,7777,8057,91676,3.17
language,19787,0,135,3.96
is_part_of_series,0,0,2,0.00


### 6.1 Cuantificar población y exclusiones por páginas

El filtro de calificación define la población destacada. Para comparar extensión se consideran provisionalmente páginas entre 1 y 5.000: el rango permite aislar ausencias, ceros y extremos que requieren revisión; no implica que todo valor dentro del rango sea correcto. Los registros sin páginas válidas se contabilizan aparte y no se asignan silenciosamente a un área.

In [28]:
PAGINAS_MIN_VALIDAS, PAGINAS_MAX_VALIDAS = 1, 5000
mascara_destacados = df_original["average_rating"].ge(3.5)
destacados = df_original.loc[mascara_destacados].copy()
mascara_paginas_validas = destacados["pages"].between(
    PAGINAS_MIN_VALIDAS, PAGINAS_MAX_VALIDAS, inclusive="both"
)
destacados_con_paginas_validas = destacados.loc[mascara_paginas_validas].copy()

conteos_poblacion = pd.DataFrame({
    "registros": [
        len(df_original),
        int(df_original["average_rating"].isna().sum()),
        int((df_original["average_rating"].eq(0) & df_original["ratings_count"].eq(0)).sum()),
        int((df_original["average_rating"].eq(0) & df_original["ratings_count"].gt(0)).sum()),
        len(destacados),
        int(destacados["pages"].isna().sum()),
        int(destacados["pages"].le(0).sum()),
        int(destacados["pages"].gt(PAGINAS_MAX_VALIDAS).sum()),
        len(destacados_con_paginas_validas),
    ]
}, index=[
    "Dataset original",
    "Calificación faltante",
    "Calificación cero y ratings_count cero",
    "Calificación cero con ratings_count positivo",
    "Calificación >= 3,5",
    "Destacados con páginas faltantes",
    "Destacados con páginas <= 0",
    "Destacados con páginas > 5.000",
    "Destacados con páginas entre 1 y 5.000",
])
conteos_poblacion["porcentaje_del_total"] = (
    conteos_poblacion["registros"] / len(df_original) * 100
).round(2)
conteos_poblacion

,registros,porcentaje_del_total
Dataset original,500000,100.00
Calificación faltante,258,0.05
Calificación cero y ratings_count cero,148219,29.64
Calificación cero con ratings_count positivo,0,0.00
"Calificación >= 3,5",276152,55.23
Destacados con páginas faltantes,22129,4.43
Destacados con páginas <= 0,18868,3.77
Destacados con páginas > 5.000,51,0.01
Destacados con páginas entre 1 y 5.000,235104,47.02


### 6.2 Sensibilidad de la clasificación por extensión

Se comparan cuatro umbrales candidatos (250, 300, 350 y 400 páginas) dentro del mismo subconjunto destacado con páginas entre 1 y 5.000. Esta tabla ayuda a evaluar cuánto cambia el reparto, pero no selecciona por sí sola el umbral final. Las obras únicas se cuentan dentro de cada grupo; una obra con varias ediciones puede aparecer en ambos, por lo que también se informa ese solapamiento.

In [29]:
UMBRALES_CANDIDATOS = (250, 300, 350, 400)
filas_sensibilidad = []
filas_solapamiento = []

for umbral in UMBRALES_CANDIDATOS:
    escenario = destacados_con_paginas_validas[["book_id", "work_id", "pages"]].copy()
    escenario["categoria_extension"] = np.where(
        escenario["pages"].le(umbral),
        "Lecturas Ágiles",
        "Obras extensas",
    )
    for categoria, grupo in escenario.groupby("categoria_extension", sort=False):
        filas_sensibilidad.append({
            "umbral_paginas": umbral,
            "categoria_extension": categoria,
            "registros_edicion": len(grupo),
            "porcentaje_registros_validos": len(grupo) / len(escenario) * 100,
            "obras_unicas_en_categoria": grupo["work_id"].nunique(),
        })
    obras_por_categoria = escenario.dropna(subset=["work_id"]).groupby("work_id")[
        "categoria_extension"
    ].nunique()
    filas_solapamiento.append({
        "umbral_paginas": umbral,
        "obras_con_ediciones_en_ambos_grupos": int(obras_por_categoria.gt(1).sum()),
    })

sensibilidad_extension = pd.DataFrame(filas_sensibilidad)
sensibilidad_extension["porcentaje_registros_validos"] = sensibilidad_extension[
    "porcentaje_registros_validos"
].round(2)
sensibilidad_extension, pd.DataFrame(filas_solapamiento)

(   umbral_paginas categoria_extension  registros_edicion  \
 0             250      Obras extensas             111917   
 1             250     Lecturas Ágiles             123187   
 2             300      Obras extensas              82325   
 3             300     Lecturas Ágiles             152779   
 4             350     Lecturas Ágiles             175788   
 5             350      Obras extensas              59316   
 6             400     Lecturas Ágiles             194193   
 7             400      Obras extensas              40911   
 
    porcentaje_registros_validos  obras_unicas_en_categoria  
 0                         47.60                      99674  
 1                         52.40                     115082  
 2                         35.02                      72828  
 3                         64.98                     141809  
 4                         74.77                     162287  
 5                         25.23                      52194  
 6             

### 6.3 Perfil descriptivo preliminar con el umbral candidato de 350 páginas

Se usa 350 solo para mostrar cómo quedaría el análisis comparativo, no como decisión aprobada. Las medianas y percentiles de `ratings_count` y `reviews_count` se calculan por registro de edición; no se suman porque esas métricas corresponden a la obra y pueden repetirse o variar entre ediciones. Las tablas de editorial e idioma cuentan registros y obras distintas, pero no representan ventas ni demanda de una librería.

In [38]:
UMBRAL_EXPLORATORIO = 350
perfil = destacados_con_paginas_validas.copy()
perfil["categoria_extension"] = np.where(
    perfil["pages"].le(UMBRAL_EXPLORATORIO),
    "Lecturas Ágiles",
    "Obras extensas",
)

resumen_por_extension = perfil.groupby("categoria_extension", sort=False).agg(
    registros_edicion=("book_id", "size"),
    obras_unicas=("work_id", "nunique"),
    mediana_paginas=("pages", "median"),
    media_calificacion=("average_rating", "mean"),
    desviacion_estandar_calificacion=("average_rating", "std"),
    mediana_calificacion=("average_rating", "median"),
    calificacion_p25=("average_rating", lambda s: s.quantile(0.25)),
    calificacion_p75=("average_rating", lambda s: s.quantile(0.75)),
    mediana_ratings_count_por_registro=("ratings_count", "median"),
    p90_ratings_count_por_registro=("ratings_count", lambda s: s.quantile(0.90)),
    mediana_reviews_count_por_registro=("reviews_count", "median"),
    p90_reviews_count_por_registro=("reviews_count", lambda s: s.quantile(0.90)),
    registros_en_series=("is_part_of_series", "sum"),
    proporcion_registros_en_series=("is_part_of_series", "mean"),
    publisher_faltante=("publisher", lambda s: s.isna().sum()),
    language_faltante=("language", lambda s: s.isna().sum()),
).round(2)
resumen_por_extension

,registros_edicion,obras_unicas,mediana_paginas,media_calificacion,desviacion_estandar_calificacion,mediana_calificacion,calificacion_p25,calificacion_p75,mediana_ratings_count_por_registro,p90_ratings_count_por_registro,mediana_reviews_count_por_registro,p90_reviews_count_por_registro,registros_en_series,proporcion_registros_en_series,publisher_faltante,language_faltante
categoria_extension,,,,,,,,,,,,,,,,
Lecturas Ágiles,175788,162287,192.0,4.07,0.40,4.0,3.78,4.25,17.0,2777.0,2.0,204.0,32296,0.18,3163,4269
Obras extensas,59316,52194,453.0,4.08,0.37,4.0,3.82,4.25,51.0,13940.5,5.0,787.0,12207,0.21,1219,1307


#### Editoriales e idiomas más frecuentes por grupo

Los nombres se muestran tal como aparecen en la fuente; antes de una tabla final habría que decidir y documentar una normalización conservadora. Se excluyen de estos rankings los valores nulos o vacíos, pero se reportan sus cantidades en el perfil anterior. Los porcentajes de esta tabla usan como denominador los registros con esa categoría informada dentro del grupo.

In [31]:
filas_top_categorias = []
for categoria_extension, grupo in perfil.groupby("categoria_extension", sort=False):
    for campo, nombre_variable in [("publisher", "editorial"), ("language", "idioma")]:
        informado = grupo.loc[grupo[campo].notna(), ["book_id", "work_id", campo]].copy()
        informado[campo] = informado[campo].astype("string").str.strip()
        informado = informado.loc[informado[campo].ne("")]
        conteo = (
            informado.groupby(campo, dropna=False)
            .agg(registros_edicion=("book_id", "size"), obras_unicas=("work_id", "nunique"))
            .sort_values(["registros_edicion", "obras_unicas"], ascending=False)
            .head(10)
        )
        for valor, fila in conteo.iterrows():
            filas_top_categorias.append({
                "categoria_extension": categoria_extension,
                "variable": nombre_variable,
                "valor_fuente": str(valor),
                "registros_edicion": int(fila["registros_edicion"]),
                "porcentaje_de_informados": fila["registros_edicion"] / len(informado) * 100,
                "obras_unicas": int(fila["obras_unicas"]),
            })

top_editoriales_idiomas = pd.DataFrame(filas_top_categorias)
top_editoriales_idiomas["porcentaje_de_informados"] = top_editoriales_idiomas[
    "porcentaje_de_informados"
].round(2)
top_editoriales_idiomas

,categoria_extension,variable,valor_fuente,registros_edicion,porcentaje_de_informados,obras_unicas
0,Lecturas Ágiles,editorial,Routledge,2296,1.34,2246
1,Lecturas Ágiles,editorial,Oxford University Press,1499,0.88,1457
2,Lecturas Ágiles,editorial,Cambridge University Press,1366,0.80,1312
3,Lecturas Ágiles,editorial,HarperCollins,819,0.48,791
4,Lecturas Ágiles,editorial,iUniverse,628,0.37,617
5,Lecturas Ágiles,editorial,Scholastic,617,0.36,614
6,Lecturas Ágiles,editorial,Springer,597,0.35,589
7,Lecturas Ágiles,editorial,Wiley,517,0.30,508
8,Lecturas Ágiles,editorial,Kessinger Publishing,514,0.30,510
9,Lecturas Ágiles,editorial,Palgrave Macmillan,510,0.30,502


### 6.4 Control de ediciones repetidas por obra

Una obra puede tener varias ediciones y aparecer en ambos grupos de páginas. El siguiente control cuantifica cuántas obras tienen más de un registro y en cuántas cambian las métricas agregadas. Esto justifica presentar conteos de ediciones y obras por separado, y evita sumar las métricas de lectores por edición como si fueran independientes.

In [32]:
con_id_obra = df_original.dropna(subset=["work_id"])
registros_por_obra = con_id_obra.groupby("work_id").size()
metricas_por_obra = con_id_obra.groupby("work_id")[[
    "average_rating", "ratings_count", "reviews_count"
]].nunique(dropna=True)

control_obras = pd.DataFrame({
    "medición": [
        "Registros con work_id",
        "Obras únicas",
        "Obras con más de un registro",
        "Obras repetidas con distinto average_rating",
        "Obras repetidas con distinto ratings_count",
        "Obras repetidas con distinto reviews_count",
        "Obras con ediciones en ambos grupos al umbral 350",
    ],
    "cantidad": [
        len(con_id_obra),
        int(registros_por_obra.size),
        int(registros_por_obra.gt(1).sum()),
        int(metricas_por_obra.loc[registros_por_obra.gt(1), "average_rating"].gt(1).sum()),
        int(metricas_por_obra.loc[registros_por_obra.gt(1), "ratings_count"].gt(1).sum()),
        int(metricas_por_obra.loc[registros_por_obra.gt(1), "reviews_count"].gt(1).sum()),
        int(perfil.dropna(subset=["work_id"]).groupby("work_id")["categoria_extension"].nunique().gt(1).sum()),
    ],
})
control_obras

,medición,cantidad
0,Registros con work_id,499747
1,Obras únicas,461678
2,Obras con más de un registro,24753
3,Obras repetidas con distinto average...,1552
4,Obras repetidas con distinto ratings...,12181
5,Obras repetidas con distinto reviews...,7242
6,Obras con ediciones en ambos grupos ...,2506


### 6.5 Decisiones metodológicas adoptadas y pendientes

Se adopta provisionalmente un umbral de 350 páginas: `Lecturas Ágiles` hasta 350 y `Obras extensas` por encima de 350, dentro del rango de 1 a 5.000 páginas. Es una decisión operativa cercana al percentil 75 de páginas del subconjunto válido, no una frontera natural.

La preparación conservará los registros elegibles aunque falte editorial o idioma, y mantendrá conteos de ediciones y obras. Las métricas de lectores se describirán como distribuciones por registro; no se sumarán entre ediciones. El proyecto informa patrones de Goodreads y no infiere ventas ni demanda de una librería.

## 7. Preparación y enriquecimiento para responder las preguntas

Se adopta el umbral provisional acordado de **350 páginas**. La población preparada incluye registros con `average_rating >= 3.5` y páginas entre 1 y 5.000. Se conservan nulos de editorial, idioma y métricas de lectores; no se eliminan duplicados por título y se mantienen `book_id` y `work_id`. La editorial y el idioma se limpian solo de espacios externos, preservando los valores originales.

El enriquecimiento deriva `categoria_extension` y `cantidad_series`. La tabla de resultados distingue registros de edición y obras únicas; las obras que tienen ediciones a ambos lados del umbral pueden contarse en ambos grupos.

In [33]:
UMBRAL_PAGINAS = 350
columnas_fuente = [
    "book_id", "work_id", "title", "pages", "average_rating",
    "ratings_count", "reviews_count", "publisher", "language",
    "is_part_of_series", "series", "format",
]
mascara_preparacion = (
    df_original["average_rating"].ge(3.5)
    & df_original["pages"].between(PAGINAS_MIN_VALIDAS, PAGINAS_MAX_VALIDAS, inclusive="both")
)
df_preparado = df_original.loc[mascara_preparacion, columnas_fuente].copy()

# Conservar los valores originales para rastrear la limpieza de categorías.
for campo in ["publisher", "language"]:
    df_preparado[f"{campo}_original"] = df_preparado[campo]
    df_preparado[campo] = df_preparado[campo].astype("string").str.strip()
    df_preparado[campo] = df_preparado[campo].replace("", pd.NA)

df_preparado["categoria_extension"] = np.where(
    df_preparado["pages"].le(UMBRAL_PAGINAS),
    "Lecturas \u00c1giles",
    "Obras extensas",
)
df_preparado["cantidad_series"] = df_preparado["series"].map(
    lambda valor: len(valor) if isinstance(valor, (list, tuple, np.ndarray)) else 0
)
df_preparado = df_preparado.drop(columns="series")
df_preparado = df_preparado[[
    "book_id", "work_id", "title", "pages", "categoria_extension",
    "average_rating", "ratings_count", "reviews_count",
    "publisher_original", "publisher", "language_original", "language",
    "is_part_of_series", "cantidad_series", "format",
]]
print(f"DataFrame preparado: {len(df_preparado):,} filas y {len(df_preparado.columns)} columnas")
df_preparado.head()

DataFrame preparado: 235,104 filas y 15 columnas


,book_id,work_id,title,pages,categoria_extension,average_rating,ratings_count,reviews_count,publisher_original,publisher,language_original,language,is_part_of_series,cantidad_series,format
0,7282453,44066165.0,La doctrina secreta. Vol I. Cosmogen...,319.0,Lecturas Ágiles,4.16,255.0,22.0,Editorial Kier,Editorial Kier,Spanish; Castilian,Spanish; Castilian,True,1,Paperback
3,2027361,580945.0,Personality and Intellectual Competence,214.0,Lecturas Ágiles,3.88,8.0,0.0,Psychology Press,Psychology Press,English,English,False,0,Hardcover
7,183242,177086.0,Detik-Detik Diri Di Daerah Daif,191.0,Lecturas Ágiles,4.38,77.0,17.0,"Dewan Bahasa dan Pustaka, Kementeria...","Dewan Bahasa dan Pustaka, Kementeria...",Malay,Malay,False,0,Paperback
11,2501455,2508820.0,فوضى الحواس,375.0,Obras extensas,3.78,20120.0,1462.0,دار الآداب,دار الآداب,Arabic,Arabic,True,1,Paperback
13,985576,971071.0,"Next door, down the road, around the...",188.0,Lecturas Ágiles,5.00,1.0,0.0,Doubleday,Doubleday,English,English,False,0,Paperback


### 7.1 Evidencia de cambios y controles de calidad

La comparación muestra dimensiones antes/después, tipo de dato y valores nulos por columna. Las columnas derivadas no existen en la fuente; las categorías sin editorial o idioma no se excluyen. La regla de páginas retira del análisis por extensión los registros destacados con páginas ausentes, cero/no positivas o mayores de 5.000; sus cantidades quedan separadas en el control.

In [34]:
comparacion_dimensiones = pd.DataFrame(
    [df_original.shape, df_preparado.shape],
    index=["Original sin modificar", "Preparado y enriquecido"],
    columns=["filas", "columnas"],
)
columnas_comparacion = list(dict.fromkeys([*df_original.columns, *df_preparado.columns]))
comparacion_tipos = pd.DataFrame({
    "tipo_original": [str(df_original[c].dtype) if c in df_original else pd.NA for c in columnas_comparacion],
    "tipo_preparado": [str(df_preparado[c].dtype) if c in df_preparado else pd.NA for c in columnas_comparacion],
}, index=columnas_comparacion)
comparacion_nulos = pd.DataFrame({
    "nulos_original": [int(df_original[c].isna().sum()) if c in df_original else pd.NA for c in columnas_comparacion],
    "nulos_preparado": [int(df_preparado[c].isna().sum()) if c in df_preparado else pd.NA for c in columnas_comparacion],
}, index=columnas_comparacion)
control_filas = pd.DataFrame({
    "etapa": ["Dataset original", "Calificación >= 3,5", "Páginas válidas 1-5.000 / preparados"],
    "registros": [len(df_original), int(df_original["average_rating"].ge(3.5).sum()), len(df_preparado)],
})
control_filas["excluidos_desde_etapa_anterior"] = control_filas["registros"].shift(1) - control_filas["registros"]
control_filas.loc[0, "excluidos_desde_etapa_anterior"] = 0
control_calidad_paginas = pd.DataFrame({
    "motivo": ["Páginas faltantes entre destacados", "Páginas <= 0 entre destacados", "Páginas > 5.000 entre destacados"],
    "registros": [int(destacados["pages"].isna().sum()), int(destacados["pages"].le(0).sum()), int(destacados["pages"].gt(PAGINAS_MAX_VALIDAS).sum())],
})
assert df_original.shape == (500000, 24)
assert df_preparado["book_id"].is_unique
assert len(df_preparado) == int(control_filas.loc[2, "registros"])
comparacion_dimensiones, control_filas, control_calidad_paginas, comparacion_tipos, comparacion_nulos

(                          filas  columnas
 Original sin modificar   500000        24
 Preparado y enriquecido  235104        15,
                                   etapa  registros  \
 0                      Dataset original     500000   
 1                   Calificación >= 3,5     276152   
 2  Páginas válidas 1-5.000 / preparados     235104   
 
    excluidos_desde_etapa_anterior  
 0                             0.0  
 1                        223848.0  
 2                         41048.0  ,
                                motivo  registros
 0  Páginas faltantes entre destacados      22129
 1       Páginas <= 0 entre destacados      18868
 2    Páginas > 5.000 entre destacados         51,
                             tipo_original tipo_preparado
 title                                 str            str
 contributors                       object            NaN
 first_author                          str            NaN
 description                           str            NaN
 genres 

### 7.2 Tablas que responden las preguntas de negocio

La primera tabla describe el reparto y las métricas por grupo de extensión. Los rankings de editorial e idioma cuentan registros de edición y obras únicas, y presentan mediana y percentil 90 de valoración/participación por registro, no totales de lectores. Los nulos se reportan y no se interpretan como una categoría conocida. La tabla de series mide registros y obras con indicador de pertenencia.

In [37]:
orden_categorias = ["Lecturas \u00c1giles", "Obras extensas"]
perfil_resultados = df_preparado.groupby("categoria_extension", sort=False, observed=True).agg(
    registros_edicion=("book_id", "size"),
    obras_unicas=("work_id", "nunique"),
    mediana_paginas=("pages", "median"),
    media_calificacion=("average_rating", "mean"),
    desviacion_estandar_calificacion=("average_rating", "std"),
    mediana_calificacion=("average_rating", "median"),
    calificacion_p25=("average_rating", lambda s: s.quantile(0.25)),
    calificacion_p75=("average_rating", lambda s: s.quantile(0.75)),
    mediana_ratings_count_por_registro=("ratings_count", "median"),
    p90_ratings_count_por_registro=("ratings_count", lambda s: s.quantile(0.90)),
    mediana_reviews_count_por_registro=("reviews_count", "median"),
    p90_reviews_count_por_registro=("reviews_count", lambda s: s.quantile(0.90)),
    registros_en_series=("is_part_of_series", "sum"),
    proporcion_en_series=("is_part_of_series", "mean"),
).reindex(orden_categorias).round(2)
perfil_resultados["porcentaje_registros"] = (
    perfil_resultados["registros_edicion"] / len(df_preparado) * 100
).round(2)

def crear_ranking(campo: str) -> pd.DataFrame:
    filas = []
    for categoria, grupo in df_preparado.groupby("categoria_extension", sort=False, observed=True):
        informados = grupo.loc[grupo[campo].notna()]
        resumen = informados.groupby(campo, dropna=False).agg(
            registros_edicion=("book_id", "size"),
            obras_unicas=("work_id", "nunique"),
            mediana_calificacion=("average_rating", "median"),
            mediana_ratings_count_por_registro=("ratings_count", "median"),
            mediana_reviews_count_por_registro=("reviews_count", "median"),
        ).sort_values(["registros_edicion", "obras_unicas"], ascending=False).head(10)
        for valor, fila in resumen.iterrows():
            filas.append({
                "categoria_extension": categoria,
                campo: str(valor),
                "registros_edicion": int(fila["registros_edicion"]),
                "porcentaje_registros_con_valor": fila["registros_edicion"] / len(informados) * 100,
                "obras_unicas": int(fila["obras_unicas"]),
                "mediana_calificacion": fila["mediana_calificacion"],
                "mediana_ratings_count_por_registro": fila["mediana_ratings_count_por_registro"],
                "mediana_reviews_count_por_registro": fila["mediana_reviews_count_por_registro"],
            })
    resultado = pd.DataFrame(filas)
    if not resultado.empty:
        resultado["porcentaje_registros_con_valor"] = resultado["porcentaje_registros_con_valor"].round(2)
    return resultado

resultado_editoriales = crear_ranking("publisher")
resultado_idiomas = crear_ranking("language")
resultado_series = df_preparado.groupby("categoria_extension", sort=False, observed=True).agg(
    registros_edicion=("book_id", "size"),
    registros_en_series=("is_part_of_series", "sum"),
    porcentaje_en_series=("is_part_of_series", lambda s: s.mean() * 100),
    obras_unicas=("work_id", "nunique"),
    obras_unicas_en_series=("work_id", lambda s: s[df_preparado.loc[s.index, "is_part_of_series"]].nunique()),
).reindex(orden_categorias).round(2)
resultado_extension = perfil_resultados.reset_index()
perfil_resultados, resultado_editoriales, resultado_idiomas, resultado_series

(                     registros_edicion  obras_unicas  mediana_paginas  \
 categoria_extension                                                     
 Lecturas Ágiles                 175788        162287            192.0   
 Obras extensas                   59316         52194            453.0   
 
                      media_calificacion  desviacion_estandar_calificacion  \
 categoria_extension                                                         
 Lecturas Ágiles                    4.07                              0.40   
 Obras extensas                     4.08                              0.37   
 
                      mediana_calificacion  calificacion_p25  calificacion_p75  \
 categoria_extension                                                             
 Lecturas Ágiles                       4.0              3.78              4.25   
 Obras extensas                        4.0              3.82              4.25   
 
                      mediana_ratings_count_por_registro  

### 7.3 Exportar dataset preparado, resultados y resumen descriptivo

Cada ejecución regenera los CSV y el `.txt` en `AE2/resultados/`. El resumen describe el DataFrame preparado y enriquecido. Las exportaciones se leen de nuevo para verificar dimensiones y porcentajes; el DataFrame original queda sin modificar.

In [39]:
RUTA_RESULTADOS = ROOT / "AE2" / "resultados"
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
archivos_csv = {
    "dataset_preparado": RUTA_RESULTADOS / "libros_destacados_preparados.csv",
    "por_extension": RUTA_RESULTADOS / "resumen_por_extension.csv",
    "editoriales": RUTA_RESULTADOS / "top_editoriales_por_extension.csv",
    "idiomas": RUTA_RESULTADOS / "top_idiomas_por_extension.csv",
    "series": RUTA_RESULTADOS / "resumen_series_por_extension.csv",
}
df_preparado.to_csv(archivos_csv["dataset_preparado"], index=False, encoding="utf-8")
resultado_extension.to_csv(archivos_csv["por_extension"], index=False, encoding="utf-8")
resultado_editoriales.to_csv(archivos_csv["editoriales"], index=False, encoding="utf-8")
resultado_idiomas.to_csv(archivos_csv["idiomas"], index=False, encoding="utf-8")
resultado_series.reset_index().to_csv(archivos_csv["series"], index=False, encoding="utf-8")

metadatos_preparados = {
    "Dataset": DATASET_ID,
    "Configuración": CONFIG,
    "Split": SPLIT,
    "Revisión": REVISION,
    "Filtro": "average_rating >= 3.5",
    "Páginas incluidas": "1 a 5.000; umbral 350 páginas (provisional acordado)",
    "Unidad exportada": "Registro/edición; work_id se conserva para contar obras únicas",
    "Métricas de lectores": "Se resumen por registro; no se suman entre ediciones",
}
ruta_resumen_preparado = reportes.guardar_resumen_dataframe(
    df_preparado, RUTA_RESULTADOS / "resumen_libros_preparados.txt", metadatos_preparados
)

filas_exportadas = pd.read_csv(archivos_csv["dataset_preparado"]).shape[0]
assert filas_exportadas == len(df_preparado)
assert abs(perfil_resultados["porcentaje_registros"].sum() - 100) < 0.01
assert sum(int(df_preparado["categoria_extension"].eq(c).sum()) for c in orden_categorias) == len(df_preparado)
for nombre, ruta in archivos_csv.items():
    print(f"{nombre}: {ruta.relative_to(ROOT).as_posix()}")
print(f"resumen: {ruta_resumen_preparado.relative_to(ROOT).as_posix()}")
print(f"Verificación: {filas_exportadas:,} filas exportadas; porcentajes por extensión suman {perfil_resultados['porcentaje_registros'].sum():.2f}%.")
print(f"Participación en esta tabla: {perfil_resultados.iloc[0]['porcentaje_registros']:.2f}% lecturas ágiles y {perfil_resultados.iloc[1]['porcentaje_registros']:.2f}% obras extensas.")
print("ratings_count y reviews_count se reportan como distribuciones por registro; no se interpretan como totales de personas únicas.")

dataset_preparado: AE2/resultados/libros_destacados_preparados.csv
por_extension: AE2/resultados/resumen_por_extension.csv
editoriales: AE2/resultados/top_editoriales_por_extension.csv
idiomas: AE2/resultados/top_idiomas_por_extension.csv
series: AE2/resultados/resumen_series_por_extension.csv
resumen: AE2/resultados/resumen_libros_preparados.txt
Verificación: 235,104 filas exportadas; porcentajes por extensión suman 100.00%.
Participación en esta tabla: 74.77% lecturas ágiles y 25.23% obras extensas.
ratings_count y reviews_count se reportan como distribuciones por registro; no se interpretan como totales de personas únicas.


### 7.4 Alcance de las conclusiones

Los rankings describen presencia de ediciones en esta revisión del conjunto, no ventas ni demanda local. Un `work_id` puede aparecer en ambos grupos porque sus ediciones pueden tener distinta extensión; por ello las obras únicas de cada grupo no necesariamente suman el total global. Las métricas de lectores son distribuciones por registro y no se suman. Las decisiones de compra o espacio requieren contrastar estos hallazgos con inventario, ventas y dimensiones físicas de la librería.